## 1. Overview of product types and their unique attributes
The purpose for this analysis, is to do a spectral clustering, simmilar to 5_spectral.ipynb. However, we focus this analysis on whether the clusters are simmilar within a each product type. 


`products_en.json` has one row per product with its `product_type` and its list of feature names.Below, theres a table with of products that counts products and unique features per type, to show which types have enough features to form meaningful clusters.

In [1]:
import pandas as pd

products = pd.read_json("data/products_en.json")

summary = (
    products.groupby("product_type")
    .agg(products=("product_id", "count"),
         unique_attributes=("titles", lambda s: len(set(a for lst in s for a in lst))))
    .sort_values("unique_attributes", ascending=False)
)
summary

,products,unique_attributes
product_type,,
Cycling Wear Tops,42,93
Shoe Covers,21,70
Headwear,24,65
Cycling Wear Bottoms,31,62
Gloves Cycling Winter,15,58
Socks,25,56
Base Layers,21,40
Gloves Cycling Summer Long,6,20
Gloves Cycling Summer Short,7,20


## 2. Filter similarity matrix to one product type and set parameters

The similarity matrix was built once on the full catalog. Here it is **sliced** to the features of `CATEGORY`. This can manually be changed for each run to see outcomes of the different product types. The graph, eigenvectors, number of clusters and k-means is recomputed on the subset. 

Parameters that depend on the number of features `n` are scaled down: 8 neighbours instead of 15, and k searched from 3 to n/5 instead of 20 to 80.

Steps:
1. Load the full matrix and feature names.
2. Keep only the feature names used by products of `CATEGORY`.
3. Check that the filtered matrix is square and symmetric.
4. Set the parameters that depend on the number of features `n`. This must happen after filtering. The full-run values (15 neighbours, 20 to 80 clusters) are too large for 93 features, so they are scaled down: `K_NEIGHBORS = 8`, `K_MIN = 3`, `K_MAX = n // 5`. 

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.cluster import KMeans
from scipy.sparse.csgraph import connected_components

SEED = 0
CATEGORY = "Headwear"   # Change for each run for the category you want to analyze

# Load the full similarity matrix
with np.load("data/similarity.npz") as data:
    S = data["similarity"]
    attributes = data["titles"].tolist()
attributes_full = list(attributes)

# Filter to attributes used by products of this category
cat_df = products[products["product_type"] == CATEGORY]
category_attrs = set(cat_df["titles"].explode())

missing = category_attrs - set(attributes_full)
print(f"Titles in products but not in similarity.npz: {len(missing)}", list(missing)[:5])

keep = [i for i, a in enumerate(attributes_full) if a in category_attrs]
S = S[np.ix_(keep, keep)]
attributes = [attributes_full[i] for i in keep]
n = len(attributes)

# Checks on the filtered matrix
assert S.shape == (n, n)
assert np.allclose(S, S.T), "Similarity matrix must be symmetric"
print(f"{CATEGORY}: {len(cat_df)} products, {n} attributes")

# Settings that depend on n (must come after the filter)
K_NEIGHBORS = 8
K_MIN = 3
K_MAX = n // 5
N_EIGEN = K_MAX + 1

print(f"K_NEIGHBORS={K_NEIGHBORS}, K_MIN={K_MIN}, K_MAX={K_MAX}")

Titles in products but not in similarity.npz: 0 []
Headwear: 24 products, 65 attributes
K_NEIGHBORS=8, K_MIN=3, K_MAX=13


## 3. Build graph, Laplacian and eigengap to estimate the number of clusters

1. **Affinity graph:** each feature is linked to its most similar features. An edge is kept if it is in the top `K_NEIGHBORS` of either endpoint, which keeps the graph symmetric.
2. **Normalized Laplacian:** L = I - D^(-1/2) W D^(-1/2). Rescaling by degree stops highly connected features from dominating.
3. **Eigenvectors:** those with the smallest eigenvalues describe how the graph splits into groups.
4. **Eigengap heuristic:** the number of clusters is taken at the largest jump between consecutive eigenvalues, searched between `K_MIN` and `K_MAX`.
5. **Connectivity check:** each disconnected component adds an eigenvalue of 0 and distorts the eigengap. If there is more than one component, `K_NEIGHBORS` is increased.

In [3]:
def build_affinity(matrix, k_neighbors):
    affinity = matrix.copy()
    np.fill_diagonal(affinity, 0.0)
    threshold = np.sort(affinity, axis=1)[:, -k_neighbors][:, None]
    return np.where((affinity >= threshold) | (affinity >= threshold.T), affinity, 0.0)


def spectral_embedding(affinity, n_eigen):
    degree = affinity.sum(axis=1)
    inv_sqrt = 1 / np.sqrt(np.maximum(degree, 1e-12))
    L = np.eye(len(affinity)) - affinity * np.outer(inv_sqrt, inv_sqrt)
    eigenvalues, eigenvectors = np.linalg.eigh(L)
    return eigenvalues[:n_eigen], eigenvectors[:, :n_eigen]


affinity = build_affinity(S, K_NEIGHBORS)
eigenvalues, eigenvectors = spectral_embedding(affinity, N_EIGEN)

gaps = np.diff(eigenvalues)
cluster_count = int(np.argmax(gaps[K_MIN - 1:K_MAX]) + K_MIN)

n_comp, _ = connected_components(affinity > 0, directed=False)
print(f"Edges kept: {np.count_nonzero(np.triu(affinity))} (max possible {n * K_NEIGHBORS})")   # NEW: max possible
print(f"Connected components: {n_comp}")
print(f"Eigengap suggests {cluster_count} clusters")
print("Gap per k:", {k: round(float(gaps[k - 1]), 4) for k in range(K_MIN, K_MAX + 1)})          # NEW

Edges kept: 348 (max possible 520)
Connected components: 1
Eigengap suggests 4 clusters
Gap per k: {3: 0.0128, 4: 0.1704, 5: 0.0177, 6: 0.0964, 7: 0.1316, 8: 0.0211, 9: 0.068, 10: 0.0882, 11: 0.0038, 12: 0.014, 13: 0.0655}


## 4. Cluster the features and name each cluster

The first k eigenvectors are row-normalized and grouped with k-means (Ng-Jordan-Weiss). Each cluster is named by its medoid, the feature with the highest total similarity to the other features in its cluster.

In [4]:
embedding = eigenvectors[:, :cluster_count]
embedding = embedding / np.maximum(np.linalg.norm(embedding, axis=1, keepdims=True), 1e-12)

labels = KMeans(n_clusters=cluster_count, n_init=10, random_state=SEED).fit_predict(embedding)

results = pd.DataFrame({"Attribute": attributes, "Cluster": labels})
cluster_counts = results["Cluster"].value_counts().sort_index()
print("Cluster sizes:")
print(cluster_counts.to_string())

def group_name(label):
    indices = np.where(labels == label)[0]
    within_cluster = S[np.ix_(indices, indices)]
    return attributes[indices[within_cluster.sum(axis=1).argmax()]]

cluster_names = {l: group_name(l) for l in sorted(results["Cluster"].unique())}
results["Cluster name"] = results["Cluster"].map(cluster_names)

print("\nCLUSTER NAMES")
for c in sorted(cluster_names):
    print(f"Cluster {c}: {cluster_names[c]} ({cluster_counts.loc[c]} attributes)")

display(results.sort_values(["Cluster", "Attribute"])[["Cluster", "Cluster name", "Attribute"]].reset_index(drop=True))

Cluster sizes:
Cluster
0    19
1    24
2    15
3     7

CLUSTER NAMES
Cluster 0: Under helmet fit (19 attributes)
Cluster 1: Windproof protection (24 attributes)
Cluster 2: Breathable and moisture wicking (15 attributes)
Cluster 3: Visibility (7 attributes)


,Cluster,Cluster name,Attribute
0,0,Under helmet fit,Adaptable
1,0,Under helmet fit,Adjustable strap closure
2,0,Under helmet fit,Available in two sizes
3,0,Under helmet fit,Casual wear
4,0,Under helmet fit,Compact and lightweight
...,...,...,...
60,3,Visibility,Reflective details
61,3,Visibility,Reversible brim
62,3,Visibility,Stay visible
63,3,Visibility,Subtle style


### 4b. Check the choice of k
For each k from `K_MIN` to `K_MAX` we print the eigengap, the silhouette score and the size of the smallest cluster. A clear gap at the chosen k supports the choice. A tiny smallest cluster at larger k signals over-splitting. If we override the eigengap, we must say so.

In [5]:
from sklearn.metrics import silhouette_score

D_chk = 1 - S
np.fill_diagonal(D_chk, 0)

print(f"{'k':>3} {'gap':>8} {'silhouette':>11} {'smallest':>9}")
for k_ in range(K_MIN, K_MAX + 1):
    U = eigenvectors[:, :k_]
    U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
    lab = KMeans(n_clusters=k_, n_init=10, random_state=SEED).fit_predict(U)
    sil = silhouette_score(D_chk, lab, metric="precomputed")
    print(f"{k_:>3} {gaps[k_ - 1]:>8.4f} {sil:>11.3f} {min(Counter(lab).values()):>9}")

  k      gap  silhouette  smallest
  3   0.0128       0.114         7
  4   0.1704       0.139         7
  5   0.0177       0.143         7
  6   0.0964       0.163         7
  7   0.1316       0.189         7
  8   0.0211       0.194         6
  9   0.0680       0.197         4
 10   0.0882       0.181         4
 11   0.0038       0.195         3
 12   0.0140       0.206         3
 13   0.0655       0.221         2


## 5. Compare with the general run
For each filtered cluster we count which general-run clusters its features came from. If most come from one general cluster, the theme is **stable**. If they come from several, those themes **merged** in this product type.

In [6]:
full = pd.read_csv("data/full_spectral_labels.csv")
print("Columns in file:", full.columns.tolist())   # must include 'Attribute' and 'Cluster name'

full_name_of = dict(zip(full["Attribute"], full["Cluster"]))

compare = pd.DataFrame({
    "Feature": attributes,
    "Filtered cluster": [cluster_names[l] for l in labels],
    "Full-run cluster": [full_name_of[a] for a in attributes],
})

print(f"{CATEGORY}: for each filtered cluster, where its features came from in the full run\n")
for name in compare["Filtered cluster"].unique():
    sub = compare[compare["Filtered cluster"] == name]
    print(f"{name} ({len(sub)} features)")
    for full_name, count in sub["Full-run cluster"].value_counts().head(4).items():
        print(f"    {count:>2} from full-run '{full_name}'")

Columns in file: ['Attribute', 'Cluster']
Headwear: for each filtered cluster, where its features came from in the full run

Windproof protection (24 features)
     6 from full-run '10'
     5 from full-run '12'
     4 from full-run '20'
     3 from full-run '4'
Breathable and moisture wicking (15 features)
     8 from full-run '6'
     4 from full-run '2'
     2 from full-run '23'
     1 from full-run '3'
Under helmet fit (19 features)
     9 from full-run '0'
     4 from full-run '23'
     2 from full-run '1'
     1 from full-run '20'
Visibility (7 features)
     6 from full-run '18'
     1 from full-run '5'


## 6. Evaluation metrics
Silhouette (higher is better), Davies-Bouldin (lower is better) and modularity (higher is better), with the same definitions as the general run. Outliers are features with negative silhouette. These scores depend on k and on the number of features, so they are not directly comparable between runs.

In [7]:
from sklearn.metrics import silhouette_score, silhouette_samples

D = 1 - S
np.fill_diagonal(D, 0)
G = S.copy()
np.fill_diagonal(G, 0)

def clustroid(lab):
    out = {}
    for c in np.unique(lab):
        members = np.where(lab == c)[0]
        out[c] = members[D[np.ix_(members, members)].sum(axis=1).argmin()]
    return out

def davies_bouldin(lab):
    centre = clustroid(lab)
    clusters = list(centre)
    spread = np.array([D[lab == c, centre[c]].mean() for c in clusters])
    centres = [centre[c] for c in clusters]
    separation = D[np.ix_(centres, centres)] + np.diag([np.inf] * len(clusters))
    ratios = (spread[:, None] + spread[None, :]) / separation
    return float(ratios.max(axis=1).mean())

def modularity(lab):
    degree = G.sum(axis=1)
    two_m = degree.sum()
    same = lab[:, None] == lab[None, :]
    return float(((G - np.outer(degree, degree) / two_m) * same).sum() / two_m)

sil_samples = silhouette_samples(D, labels, metric="precomputed")

print(f"{CATEGORY}: n = {n} features, k = {cluster_count}")
print(f"  silhouette      {silhouette_score(D, labels, metric='precomputed'):.3f}")
print(f"  davies_bouldin  {davies_bouldin(labels):.3f}")
print(f"  modularity      {modularity(labels):.3f}")
print(f"  outliers (negative silhouette): {(sil_samples < 0).sum()} of {n}")
print(f"  smallest cluster: {min(Counter(labels).values())}, largest: {max(Counter(labels).values())}")

Headwear: n = 65 features, k = 4
  silhouette      0.139
  davies_bouldin  1.530
  modularity      0.303
  outliers (negative silhouette): 1 of 65
  smallest cluster: 7, largest: 24


## 7. Visualize the clusters

t-SNE projects the features into 2D using the precomputed distances (1 - similarity). It is only used for **visualization** and does not change the clusters. Perplexity is reduced from 30 to at most 15 because there are far fewer points than in the full run. The eight largest clusters are coloured, the other groups are grey and singletons are hollow. The bar chart shows the size of every cluster.

t-SNE preserves local neighbourhoods, not global distances, so only the tightness and separation of clusters should be interpreted, not how far apart they are.

In [8]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from IPython.display import HTML, display

HIGHLIGHT_GROUPS = 8
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK = "#8c8c86", "#c9c8c0", "#1f2933"

# t-SNE makes a 2D picture from similarities; it does not change the clusters.
distance = np.clip(1 - S, 0, None)
np.fill_diagonal(distance, 0)
coordinates = TSNE(n_components=2, metric="precomputed", init="random",
                   perplexity=min(15, (n - 1) // 3), random_state=42).fit_transform(distance)
ranked = sorted(cluster_names, key=lambda label: -cluster_counts[label])
highlighted = ranked[:HIGHLIGHT_GROUPS]
figure = go.Figure()


def add_group(indices, name, color, size=7, hollow=False):
    if len(indices) == 0:
        return
    figure.add_trace(go.Scatter(
        x=coordinates[indices, 0], y=coordinates[indices, 1], mode="markers",
        name=name, text=[attributes[i] for i in indices],
        customdata=[cluster_names.get(labels[i], "Singleton") for i in indices],
        marker=dict(size=size, color="white" if hollow else color,
                    line=dict(width=1.5 if hollow else 1, color=color if hollow else "white")),
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    ))


singletons = np.where(np.array([cluster_counts[label] == 1 for label in labels]))[0]
others = np.where(np.array([
    cluster_counts[label] > 1 and label not in highlighted for label in labels
]))[0]
add_group(singletons, f"Singletons ({len(singletons)})", LIGHT_GRAY, hollow=True)
add_group(others, f"Other groups ({len(ranked) - len(highlighted)})", GRAY)
for color, label in zip(PALETTE, highlighted):
    add_group(np.where(labels == label)[0],
              f"{cluster_names[label]} ({cluster_counts[label]})", color, size=10)

figure.update_layout(
    title=dict(text=f"Feature clusters: {CATEGORY} (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white", width=700, height=450,
    margin=dict(l=15, r=330, t=45, b=15),
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK, size=10), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))


## 8. Summary across product types
This cell runs the whole pipeline once for each of the six product types with at least 50 features, using identical settings (8 neighbours, k from 3 to n/5).

For each type it does the following:
1. Select the type's features and slice the similarity matrix.
2. Build the graph, choose k with the eigengap, and run k-means.
3. Compute silhouette, Davies-Bouldin, modularity, outliers and cluster sizes. These go into the **table**. It only describes the filtered runs, so the general run's numbers (from Notebook A) are added by hand as a comparison row.
4. For each filtered cluster, look up its features in `full_spectral_labels.csv` and find the **single general cluster that supplied most of them**. The printout lists these as "x/y from general 'name'". This is the theme comparison with the general run.

In [9]:
TYPES = ["Cycling Wear Tops", "Shoe Covers", "Headwear",
         "Cycling Wear Bottoms", "Gloves Cycling Winter", "Socks"]

with np.load("data/similarity.npz") as data:
    S_all = data["similarity"]
    titles_all = data["titles"].tolist()

gen = pd.read_csv("data/full_spectral_labels.csv")
assert gen["Attribute"].tolist() == titles_all
gen_lab = gen["Cluster"].to_numpy()
gen_cluster_of = dict(zip(titles_all, gen_lab))
gen_name = {}
for c in np.unique(gen_lab):
    idx = np.where(gen_lab == c)[0]
    gen_name[c] = titles_all[idx[S_all[np.ix_(idx, idx)].sum(axis=1).argmax()]]


def scores(S_c, lab):
    D_c = 1 - S_c
    np.fill_diagonal(D_c, 0)
    G_c = S_c.copy()
    np.fill_diagonal(G_c, 0)
    cl = list(np.unique(lab))
    centre = {c: np.where(lab == c)[0][D_c[np.ix_(lab == c, lab == c)].sum(axis=1).argmin()] for c in cl}
    spread = np.array([D_c[lab == c, centre[c]].mean() for c in cl])
    centres = [centre[c] for c in cl]
    sep = D_c[np.ix_(centres, centres)] + np.diag([np.inf] * len(cl))
    db = float(((spread[:, None] + spread[None, :]) / sep).max(axis=1).mean())
    deg = G_c.sum(axis=1)
    same = lab[:, None] == lab[None, :]
    mod = float(((G_c - np.outer(deg, deg) / deg.sum()) * same).sum() / deg.sum())
    sil = float(silhouette_score(D_c, lab, metric="precomputed"))
    out = int((silhouette_samples(D_c, lab, metric="precomputed") < 0).sum())
    return sil, db, mod, out


rows, details = [], {}
for cat in TYPES:
    attrs = set(products.loc[products["product_type"] == cat, "titles"].explode())
    idx_keep = [i for i, a in enumerate(titles_all) if a in attrs]
    S_c = S_all[np.ix_(idx_keep, idx_keep)]
    names = [titles_all[i] for i in idx_keep]
    n_c = len(names)
    k_min, k_max = 3, n_c // 5

    aff = build_affinity(S_c, 8)
    ev, evec = spectral_embedding(aff, k_max + 1)
    g = np.diff(ev)
    window = g[k_min - 1:k_max]
    k = int(np.argmax(window) + k_min)
    second = float(np.sort(window)[-2])
    comps, _ = connected_components(aff > 0, directed=False)

    emb = evec[:, :k]
    emb = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)
    lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(emb)
    sil, db, mod, out = scores(S_c, lab)
    sizes = Counter(lab)

    rows.append({"type": cat, "features": n_c, "k_range": f"{k_min}-{k_max}", "k": k,
                 "best_gap": round(float(window.max()), 3), "2nd_gap": round(second, 3),
                 "components": comps, "silhouette": round(sil, 3), "davies_bouldin": round(db, 3),
                 "modularity": round(mod, 3), "outliers": out,
                 "smallest": min(sizes.values()), "largest": max(sizes.values())})

    lines = []
    for c in sorted(sizes, key=lambda c: -sizes[c]):
        members = np.where(lab == c)[0]
        medoid = names[members[S_c[np.ix_(members, members)].sum(axis=1).argmax()]]
        origin = Counter(gen_name[gen_cluster_of[names[i]]] for i in members).most_common(1)[0]
        lines.append(f"  {medoid} ({len(members)}): {origin[1]}/{len(members)} from general '{origin[0]}'")
    details[cat] = lines

display(pd.DataFrame(rows).set_index("type"))
for cat, lines in details.items():
    print(f"\n{cat}")
    print("\n".join(lines))

,features,k_range,k,best_gap,2nd_gap,components,silhouette,davies_bouldin,modularity,outliers,smallest,largest
type,,,,,,,,,,,,
Cycling Wear Tops,93,3-18,5,0.139,0.089,1,0.130,1.549,0.267,6,11,25
Shoe Covers,70,3-14,7,0.157,0.121,1,0.169,1.291,0.269,5,5,13
Headwear,65,3-13,4,0.170,0.132,1,0.139,1.530,0.303,1,7,24
Cycling Wear Bottoms,62,3-12,3,0.139,0.102,1,0.093,1.708,0.217,2,11,33
Gloves Cycling Winter,58,3-11,6,0.136,0.104,1,0.176,1.356,0.279,2,3,15
Socks,56,3-11,8,0.094,0.089,1,0.173,1.281,0.194,3,3,13



Cycling Wear Tops
  Active fit (25): 8/25 from general 'Close-fitting cut'
  Highly breathable (24): 8/24 from general 'Highly breathable'
  Windproof protection (21): 7/21 from general 'Windproof protection'
  Rear pocket storage (12): 10/12 from general 'Rear pocket storage'
  YKK Vislon 2-way zipper (11): 11/11 from general 'YKK Vislon 2-way zipper'

Shoe Covers
  Anti-abrasion tread pad (13): 10/13 from general 'Reinforced heel and toe'
  3mm neoprene (12): 6/12 from general 'Coated 3 mm neoprene'
  Aerodynamic gains (11): 7/11 from general 'Aerodynamic coverage'
  Rear zipper (11): 2/11 from general 'YKK Vislon 2-way zipper'
  Road-shoe-specific fit (10): 9/10 from general 'Road-specific fit'
  Fully waterproofed seams (8): 5/8 from general 'Fully waterproofed seams'
  Reflective details (5): 4/5 from general 'Reflective details'

Headwear
  Windproof protection (24): 6/24 from general 'Windproof protection'
  Under helmet fit (19): 9/19 from general 'Hook-and-loop closure'
  Bre